# Navigation, Sticky Reference, Stale Badges & Scratchpad (`window.craftNavRef`)

> **Phase 4 navigation & reference features:** **Minimap / Outline Rail (3.1)**, **Sticky Reference Pane (3.2)**, **Stale-Output Badges (3.4)**, and **Scratchpad / Ephemeral Cell Flag (3.5)**.

## Templates (`#| export_ft`)

In [ ]:
#| export_ft CRAFT.js
#| eval: false
TEMPLATES = {
    'nav_minimap': Aside(
        Div(
            Strong('Outline'),
            Button('×', type='button', cls='craft-modal-x', aria_label='Close outline rail', data_ref='close'),
            cls='craft-rail-head',
        ),
        Div(cls='craft-rail-list', data_ref='list'),
        id='craft-minimap', role='navigation', aria_label='Dialog outline and minimap',
    ),
    'nav_ref_pane': Aside(
        Div(cls='craft-ref-resizer', aria_hidden='true', title='Drag to resize reference pane', data_ref='resizer'),
        Div(
            Strong('Reference Pane'),
            Button('×', type='button', cls='craft-modal-x', aria_label='Close reference pane', data_ref='close'),
            cls='craft-rail-head',
        ),
        Div(cls='craft-ref-list', data_ref='list'),
        id='craft-ref-pane', role='complementary', aria_label='Pinned reference cells',
    ),
}

## Navigation, Reference, Stale & Scratchpad Runtime (`#| export_js_iife`)

In [ ]:
#| export_js_iife CRAFT.js craftNavRef
#| eval: false
window.craftNavRef?.disable?.();
if (!document.getElementById('dialog-container')) return;

const CORE = window.__craftCMCore;
const {el, qsa, on, store, mountTpl, onDomChange, notify, notifyErr, normId, getCellEl, allCells, primaryCell, cellMtype, dlgKey, deleteCellEl} = CORE;
const ac = new AbortController(), sig = {signal: ac.signal};

const MAP_OPEN_KEY = 'craft-minimap-open';
const REF_W_KEY = 'craft-ref-width';
const SCRATCH_KEY = () => `craft-scratch:${dlgKey()}`;

/* ── 3.1 Minimap / Outline Rail ────────────────────────────────────── */
let mapEl = null;
let mapOpen = store.get(MAP_OPEN_KEY, '0') === '1';

function renderMinimap() {
  if (!mapOpen) { mapEl?.root?.classList.remove('open'); return; }
  if (!mapEl) {
    mapEl = mountTpl('nav_minimap', {refs: true});
    on(mapEl.close, 'click', () => setMinimapOpen(false), sig);
    document.body.append(mapEl.root);
  }
  mapEl.root.classList.add('open');
  const cells = allCells();
  const rows = cells.map((c, idx) => {
    const cellId = normId(c.id);
    const mtype = cellMtype(c);
    const h = c.querySelector(':scope > .card-in[data-mtype="note"] .msg-content :is(h1,h2,h3,h4,h5,h6)');
    const isErr = !!c.querySelector(':scope > .card-out.craft-c2-traceback');
    const isHid = !!window.craftContext?.isHiddenFromAI?.(c);
    const isOmit = c.classList.contains('craft-omitted');
    const isSel = c.dataset.sm === 'primary' || c.dataset.sm === 'secondary';
    const label = h ? h.textContent.trim() : ((c.querySelector(':scope > .card-in .msg-content')?.textContent || '').replace(/\s+/g, ' ').trim().slice(0, 36) || `${mtype} #${idx + 1}`);
    const lvl = h ? +h.tagName.slice(1) : 0;
    return el('button', {
      type: 'button',
      cls: `craft-map-item type-${mtype}${lvl ? ` is-h is-h${lvl}` : ''}${isErr ? ' is-err' : ''}${isHid ? ' is-hid' : ''}${isOmit ? ' is-omit' : ''}${isSel ? ' is-sel' : ''}`,
      title: `${mtype}${isErr ? ' (error)' : ''}${isHid ? ' (hidden from AI)' : ''}: ${label}`,
      onClick: () => {
        const live = getCellEl(cellId);
        if (!live) return;
        live.scrollIntoView({behavior: 'smooth', block: 'center'});
        window.selectMsg?.(window.$?.(live));
        if (window.craftMobile?.active?.()) setMinimapOpen(false);
      },
    },
      el('span', {cls: 'craft-map-tick'}),
      el('span', {cls: 'craft-map-lbl'}, label),
      isErr ? el('span', {cls: 'craft-map-err', title: 'Traceback error'}, '!') : null,
    );
  });
  mapEl.list.replaceChildren(...(rows.length ? rows : [el('div', {cls: 'craft-empty-state'}, 'Empty dialog')]));
}

function setMinimapOpen(open) {
  mapOpen = !!open;
  store.set(MAP_OPEN_KEY, mapOpen ? '1' : '0');
  document.documentElement.classList.toggle('craft-minimap-open', mapOpen);
  renderMinimap();
}


### Sticky Reference Pane

In [ ]:
#| export_js_iife CRAFT.js craftNavRef
#| eval: false
/* ── 3.2 Sticky Reference Pane (Pin Cell to Side) ──────────────────── */
const pinnedRefIds = []; // max 2 cells
let refPane = null;

function sanitizeRefClone(node) {
  const clone = node.cloneNode(true);
  for (const elNode of [clone, ...qsa('*', clone)]) {
    elNode.removeAttribute?.('id');
    for (const attr of [...(elNode.attributes || [])]) {
      if (attr.name.startsWith('hx-') || attr.name.startsWith('data-hx-')) elNode.removeAttribute(attr.name);
    }
  }
  return clone;
}

function renderRefPane() {
  const valid = pinnedRefIds.filter(id => getCellEl(id));
  pinnedRefIds.length = 0; pinnedRefIds.push(...valid);
  if (!pinnedRefIds.length) {
    refPane?.root?.classList.remove('open');
    document.documentElement.classList.remove('craft-ref-open');
    return;
  }
  if (!refPane) {
    refPane = mountTpl('nav_ref_pane', {refs: true});
    const savedW = parseInt(store.get(REF_W_KEY, '360'), 10);
    if (savedW >= 220 && savedW <= 760) refPane.root.style.width = savedW + 'px';
    on(refPane.close, 'click', () => { pinnedRefIds.length = 0; renderRefPane(); }, sig);
    let dragging = false, startX = 0, startW = 360, lastW = null;
    on(refPane.resizer, 'pointerdown', e => {
      if (e.button !== 0) return;
      e.preventDefault(); dragging = true; startX = e.clientX; startW = refPane.root.offsetWidth; lastW = startW;
    }, sig);
    on(window, 'pointermove', e => {
      if (!dragging) return;
      lastW = Math.max(220, Math.min(760, startW + (startX - e.clientX)));
      refPane.root.style.width = lastW + 'px';
    }, sig);
    on(window, 'pointerup', () => {
      if (!dragging) return;
      dragging = false;
      if (lastW != null) store.set(REF_W_KEY, String(lastW));
    }, sig);
    document.body.append(refPane.root);
  }
  refPane.root.classList.add('open');
  document.documentElement.classList.add('craft-ref-open');
  refPane.list.replaceChildren(...pinnedRefIds.map(id => {
    const src = getCellEl(id);
    const body = el('div', {cls: 'craft-ref-clone'});
    for (const part of qsa(':scope > .msg-card .msg-content', src)) body.append(sanitizeRefClone(part));
    return el('div', {cls: 'craft-ref-item'},
      el('div', {cls: 'craft-ref-item-head'},
        el('span', {cls: 'craft-ref-type'}, `${cellMtype(src)} #${id.slice(0, 6)}`),
        el('button', {type: 'button', cls: 'uk-btn uk-btn-default uk-btn-xs', onClick: () => getCellEl(id)?.scrollIntoView({behavior: 'smooth', block: 'center'})}, 'Jump'),
        el('button', {type: 'button', cls: 'uk-btn uk-btn-default uk-btn-xs', onClick: () => unpinRef(id)}, 'Unpin'),
      ),
      body,
    );
  }));
}

function pinRef(cell) {
  const id = normId(cell?.id);
  if (!id) return;
  const i = pinnedRefIds.indexOf(id);
  if (i >= 0) { pinnedRefIds.splice(i, 1); renderRefPane(); return; }
  if (pinnedRefIds.length >= 2) pinnedRefIds.shift();
  pinnedRefIds.push(id);
  renderRefPane();
}

function unpinRef(id) {
  const i = pinnedRefIds.indexOf(normId(id));
  if (i >= 0) pinnedRefIds.splice(i, 1);
  renderRefPane();
}

### Stale-Output Badges & Scratchpad Flag

In [ ]:
#| export_js_iife CRAFT.js craftNavRef
#| eval: false
/* ── 3.4 Stale-Output Badges (Session-Scoped) ──────────────────────── */
const lastEditTs = new Map();
const lastRunTs = new Map();

const extractRequestIds = (params, includeContents = false) => {
  const out = new Set();
  const add = raw => {
    if (!raw) return;
    for (const part of (Array.isArray(raw) ? raw : String(raw).split(','))) {
      const cid = normId(part.trim());
      if (cid) out.add(cid);
    }
  };
  add(params?.id_);
  add(params?.ids);
  if (includeContents && params?.contents) {
    try {
      const map = typeof params.contents === 'string' ? JSON.parse(params.contents) : params.contents;
      if (map && typeof map === 'object' && !Array.isArray(map)) {
        for (const k of Object.keys(map)) add(k);
      }
    } catch {}
  }
  return [...out];
};

on(document, 'htmx:configRequest', e => {
  const path = String(e.detail?.path || '');
  const params = e.detail?.parameters;
  if (path.includes('/set_contents_')) {
    const now = Date.now();
    let changed = false;
    for (const id of extractRequestIds(params, true)) {
      const cell = getCellEl(id);
      if (cell && cellMtype(cell) === 'code') {
        lastEditTs.set(id, now);
        changed = true;
      }
    }
    if (changed) syncStaleBadges();
  } else if (path.includes('/add_runq_')) {
    const ids = extractRequestIds(params, false);
    if (!ids.length) return;
    const now = Date.now();
    for (const id of ids) lastRunTs.set(id, now);
    syncStaleBadges();
  }
}, sig);

function syncStaleBadges() {
  for (const c of allCells()) {
    const id = normId(c.id);
    const isCode = cellMtype(c) === 'code';
    const hasOut = !!c.querySelector(':scope > .card-out .msg-content')?.textContent?.trim();
    const stale = isCode && hasOut && (lastEditTs.get(id) || 0) > (lastRunTs.get(id) || 0);
    c.classList.toggle('craft-is-stale', stale);
    const hdr = c.querySelector(':scope > .card-in > .uk-card-header nav > div.grow');
    const existing = c.querySelector(':scope > .card-in .craft-stale-badge');
    if (stale && hdr && !existing) hdr.append(el('span', {cls: 'craft-stale-badge', title: 'Code edited since last run in this session'}, 'stale'));
    else if (!stale && existing) existing.remove();
  }
}

/* ── 3.5 Scratchpad / Ephemeral Cell Flag ──────────────────────────── */
const loadScratch = () => new Set((store.json(SCRATCH_KEY(), []) || []).map(normId));
const saveScratch = set => store.setJson(SCRATCH_KEY(), [...set]);

async function toggleScratch(cell) {
  if (!cell) return;
  const id = normId(cell.id);
  const set = loadScratch();
  if (set.has(id)) {
    set.delete(id);
    saveScratch(set);
  } else {
    set.add(id);
    saveScratch(set);
    await window.craftContext?.setHiddenFromAI?.(cell, true);
  }
  syncScratchBadges();
}

async function clearAllScratch() {
  const set = loadScratch();
  const targets = allCells().filter(c => set.has(normId(c.id)));
  if (!targets.length) return notifyErr('No scratch cells in this dialog.', 'craftNavRef');
  if (!confirm(`Delete ${targets.length} scratch cell(s)?`)) return;
  for (let i = targets.length - 1; i >= 0; i--) await deleteCellEl(targets[i], {tag: 'craftNavRef'});
  saveScratch(new Set());
  syncScratchBadges();
  notify(`Cleared ${targets.length} scratch cell(s).`, 'success');
}

function syncScratchBadges() {
  const set = loadScratch();
  for (const c of allCells()) {
    const isScr = set.has(normId(c.id));
    c.classList.toggle('craft-scratch-cell', isScr);
    const hdr = c.querySelector(':scope > .card-in > .uk-card-header nav > div.grow');
    const existing = c.querySelector(':scope > .card-in .craft-scratch-badge');
    if (isScr && hdr && !existing) hdr.append(el('span', {cls: 'craft-scratch-badge', title: 'Scratchpad cell (hidden from AI)'}, 'scratch'));
    else if (!isScr && existing) existing.remove();
  }
}

### Action Menu, Hotkeys & Lifecycle

In [ ]:
#| export_js_iife CRAFT.js craftNavRef
#| eval: false
/* ── Action Menu, Hotkeys & Lifecycle ──────────────────────────────── */
const offMenu = window.craftActionMenus?.registerProvider?.(ctx => {
  if (ctx.isOutput || !ctx.cell) return [];
  const isPinnedSide = pinnedRefIds.includes(ctx.id);
  const isScr = loadScratch().has(ctx.id);
  return [
    {header: 'Reference & Flags'},
    {label: isPinnedSide ? 'Unpin from side pane' : 'Pin to side pane', onClick: () => pinRef(ctx.cell)},
    {label: isScr ? 'Unmark scratch cell' : 'Mark as scratch', onClick: () => toggleScratch(ctx.cell)},
  ];
});

const reg = spec => window.__hotkeyRegister?.(spec);
reg({id: 'craft-minimap-toggle', label: 'Toggle minimap / outline rail', group: 'CRAFT · Navigation', combo: '', handler: () => { setMinimapOpen(!mapOpen); return true; }});
reg({id: 'craft-ref-pin', label: 'Pin selected cell to side reference pane', group: 'CRAFT · Navigation', combo: '', handler: () => {
  const c = primaryCell();
  if (c) pinRef(c);
  return true;
}});
reg({id: 'craft-scratch-toggle', label: 'Toggle scratchpad flag on selected cell', group: 'CRAFT · Navigation', combo: '', handler: () => {
  const c = primaryCell();
  if (c) toggleScratch(c);
  return true;
}});
reg({id: 'craft-scratch-clear', label: 'Clear all scratch cells', group: 'CRAFT · Navigation', combo: '', handler: () => { clearAllScratch(); return true; }});

const offDom = onDomChange(() => {
  if (mapOpen) renderMinimap();
  if (pinnedRefIds.length) renderRefPane();
  syncStaleBadges();
  syncScratchBadges();
}, {delay: 200, inDialog: true});

if (mapOpen) setMinimapOpen(true);
syncStaleBadges();
syncScratchBadges();

window.craftNavRef = {
  setMinimapOpen, toggleMinimap: () => setMinimapOpen(!mapOpen),
  pinRef, unpinRef, toggleScratch, clearAllScratch,
  disable() {
    ac.abort(); offDom?.(); offMenu?.();
    document.documentElement.classList.remove('craft-minimap-open', 'craft-ref-open');
    mapEl?.root?.remove(); refPane?.root?.remove();
    qsa('.craft-stale-badge, .craft-scratch-badge').forEach(b => b.remove());
    qsa('.craft-is-stale, .craft-scratch-cell').forEach(c => c.classList.remove('craft-is-stale', 'craft-scratch-cell'));
    delete window.craftNavRef;
  },
};

## Navigation, Reference & Badges CSS (`#| export_css`)

In [ ]:
#| export_css CRAFT.css
#| eval: false
/* ── Minimap, Sticky Reference Pane, Stale & Scratch Badges (16_navigation_reference) ── */
#craft-minimap, #craft-ref-pane {
  position: fixed;
  top: 40px;
  right: 0;
  bottom: 0;
  z-index: 1080;
  display: none;
  flex-direction: column;
  background: var(--craft-surface);
  border-left: 1px solid var(--craft-border);
  box-shadow: -8px 0 24px rgba(0,0,0,.35);
}
#craft-minimap { width: 220px; }
#craft-minimap.open, #craft-ref-pane.open { display: flex; }
html.craft-mobile #craft-minimap { width: min(82vw, 280px); z-index: 1240; }
html.craft-mobile #craft-ref-pane { width: min(90vw, 360px) !important; z-index: 1240; }

.craft-rail-head {
  display: flex; align-items: center; padding: 8px 10px;
  border-bottom: 1px solid var(--craft-border); font-size: 12px;
}
.craft-rail-list { overflow-y: auto; padding: 6px; display: flex; flex-direction: column; gap: 2px; }
.craft-map-item {
  display: flex; align-items: center; gap: 6px; width: 100%;
  padding: 4px 6px; border: 0; border-radius: 4px; background: transparent;
  color: var(--craft-fg); font-size: 11px; text-align: left; cursor: pointer;
}
.craft-map-item:hover, .craft-map-item.is-sel { background: var(--craft-hover); }
.craft-map-item.is-hid, .craft-map-item.is-omit { opacity: 0.45; }
.craft-map-item.is-h { font-weight: 600; color: var(--craft-fg-strong); }
.craft-map-item.is-h2 { padding-left: 12px; }
.craft-map-item.is-h3 { padding-left: 18px; }
.craft-map-tick { width: 3px; height: 12px; border-radius: 2px; flex: none; background: var(--craft-muted); }
.craft-map-item.type-code .craft-map-tick { background: var(--craft-cyan); }
.craft-map-item.type-prompt .craft-map-tick { background: var(--craft-purple); }
.craft-map-item.type-note .craft-map-tick { background: var(--craft-green); }
.craft-map-item.is-err .craft-map-tick { background: var(--craft-red); }
.craft-map-lbl { flex: 1; min-width: 0; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }
.craft-map-err { color: var(--craft-red); font-weight: 700; flex: none; }

/* Sticky Reference Pane */
#craft-ref-pane { width: 360px; }
.craft-ref-resizer {
  position: absolute; top: 0; left: -3px; bottom: 0; width: 6px; cursor: col-resize;
}
.craft-ref-list { overflow-y: auto; padding: 8px; display: flex; flex-direction: column; gap: 10px; }
.craft-ref-item { border: 1px solid var(--craft-border); border-radius: 6px; background: var(--craft-bg); overflow: hidden; }
.craft-ref-item-head {
  display: flex; align-items: center; gap: 6px; padding: 5px 8px;
  background: var(--craft-recessed); border-bottom: 1px solid var(--craft-border-soft); font-size: 10.5px;
}
.craft-ref-type { margin-right: auto; font-family: var(--craft-code-family); color: var(--craft-muted); }
.craft-ref-clone { padding: 8px; max-height: 36vh; overflow: auto; font-size: 11px; }

/* Stale & Scratchpad Badges */
.craft-stale-badge, .craft-scratch-badge {
  display: inline-block; margin-left: 6px; padding: 0 5px; border-radius: 3px;
  font-size: 9.5px; font-weight: 600; line-height: 15px; vertical-align: middle; text-transform: uppercase;
}
.craft-stale-badge {
  color: var(--craft-amber);
  border: 1px solid color-mix(in srgb, var(--craft-amber) 55%, transparent);
  background: color-mix(in srgb, var(--craft-amber) 14%, transparent);
}
.craft-scratch-badge {
  color: var(--craft-muted);
  border: 1px dashed var(--craft-border);
  background: var(--craft-recessed);
}
.editable.craft-scratch-cell > .card-in {
  border: 1px dashed var(--craft-border) !important;
  background: color-mix(in srgb, var(--craft-recessed) 60%, var(--craft-bg)) !important;
}

Verify `16_navigation_reference.ipynb` exports `window.craftNavRef`, its templates, and CSS.

In [ ]:
from craft_dev.auth import REPO_ROOT
from craft_dev.build import extract_nb_assets

assets = extract_nb_assets(REPO_ROOT/'nbs/craft/16_navigation_reference.ipynb')
js = '\n'.join(code for _, code in assets['CRAFT.js'])
css = '\n'.join(code for _, code in assets['CRAFT.css'])
for sym in ('window.craftNavRef', 'nav_minimap', 'nav_ref_pane', 'syncStaleBadges', 'clearAllScratch', 'craft-empty-state', 'deleteCellEl'):
    assert sym in js, f"Missing {sym}"
assert '#craft-minimap' in css and '#craft-ref-pane' in css and '.craft-stale-badge' in css and '.craft-scratch-badge' in css